# Intro Workshop: RAG with LlamaIndex + Ollama (Colab)

This notebook is for running the workshop in Google Colab.
You will follow a clear progression: prepare a source file, set up Ollama, compare no-RAG vs RAG, then add reranking and routing.

## Learning Goals

By the end, you will be able to:
1. Explain why RAG helps with LLM knowledge cutoffs
2. Build a simple RAG pipeline with LlamaIndex and Ollama
3. Improve retrieval quality with reranking
4. Route queries between a fact tool and a summary tool

## Prerequisites

Before running, make sure:
- You are in a Colab runtime
- Internet access is available for setup and source scraping
- (Recommended) GPU runtime is enabled

## How To Use This Notebook

Run top to bottom once, then revisit specific sections for experimentation.

### Section Map

- Section 0: Optional scrape and clean of the NHM 2024 source page
- Section 1: Colab environment setup
- Section 2: Basic RAG progression (no-RAG failure -> RAG success)
- Section 3: Advanced retrieval with reranking
- Section 4: Intent routing with two query tools
- Section 5: Optional tuning sandbox

If you restart the runtime, rerun all cells in order.

## 1) Setup Ollama In Colab

### What This Section Does

In this section you will:
- Install required Python packages
- Install and start the Ollama server in Colab
- Pull one LLM and one embedding model
- Run a quick smoke test

### 1.1 Install Python Dependencies

Install core LlamaIndex + Ollama packages for this runtime.

In [ ]:
# Colab setup: install Python dependencies
%pip install -q llama-index llama-index-llms-ollama llama-index-embeddings-ollama requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


### 1.2 Install The Ollama Runtime

Install the Ollama binary in this Colab runtime.

In [ ]:
# Colab setup: install system dependency and Ollama binary
!apt-get -qq update
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Ollama CLI found: /opt/homebrew/bin/ollama


### 1.3 Start The Ollama Service

Start Ollama at `127.0.0.1:11434` if it is not already running.

In [56]:
# Colab setup: start ollama server if needed
import os
import subprocess
import time
import urllib.request

OLLAMA_CMD = "ollama"
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"

def ollama_api_up(url="http://127.0.0.1:11434/api/tags") -> bool:
    try:
        urllib.request.urlopen(url, timeout=2).close()
        return True
    except Exception:
        return False

if not ollama_api_up():
    subprocess.Popen([OLLAMA_CMD, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(12):
        if ollama_api_up():
            break
        time.sleep(1)

if not ollama_api_up():
    raise RuntimeError("Ollama server is not reachable at http://127.0.0.1:11434")

print("Ollama server is running.")
subprocess.run([OLLAMA_CMD, "list"], check=False)

Skipping: this cell is COLAB ONLY.


### 1.4 Pull Models (First Run Only)

We pull two models:
- `llama3.2:3b` for generation
- `nomic-embed-text` for embeddings

Colab note: models are tied to the runtime and usually need re-pulling after reset.

In [ ]:
# Colab setup: pull required models
import subprocess

LLM_MODEL = "llama3.2:3b"
EMBED_MODEL = "nomic-embed-text"

for model_name in [LLM_MODEL, EMBED_MODEL]:
    print(f"Pulling {model_name}...")
    subprocess.run([OLLAMA_CMD, "pull", model_name], check=True)

print(f"Using LLM model: {LLM_MODEL}")
print(f"Using embedding model: {EMBED_MODEL}")

Ollama server is running.
]11;?\NAME                       ID              SIZE      MODIFIED       
nomic-embed-text:latest    0a109f422b47    274 MB    28 minutes ago    
llama3.2:3b                a80c4f17acd5    2.0 GB    29 minutes ago    


### 1.5 Quick Smoke Test

In [ ]:
# Colab setup: quick smoke test
import subprocess

test = subprocess.run(
    [OLLAMA_CMD, "run", LLM_MODEL, "Hello! Anyone home?"],
    capture_output=True,
    text=True,
    check=True,
)
print(test.stdout.strip())

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕

### Setup Checkpoint

If the smoke test responded correctly, your setup is ready.

If not:
- Re-run Section 1.3 (start service)
- Re-run Section 1.4 (model pull)
- Run `!ollama list` to verify models are present

## 2) Scrape And Clean The Source

Use this section to generate the workshop source file directly from the NHM article.

In [3]:
# Source prep (Colab): download prepared source text from Google Drive
from pathlib import Path

try:
    import requests
except ImportError:
    %pip install -q requests
    import requests

drive_file_id = "1A7yJs7OMnEZ3b4His28zZp24PisEYljJ"
download_url = f"https://drive.google.com/uc?export=download&id={drive_file_id}"

resp = requests.get(download_url, timeout=30)
resp.raise_for_status()
final_text = resp.text.strip()

if not final_text:
    raise ValueError("Downloaded file is empty.")

output_path = Path("data/new_species_2024.txt")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f:
    f.write(final_text)

print(f"Saved {output_path} ({len(final_text)} chars)")
print(f"Source used: {download_url}")
print(final_text[:500] + "...")

Saved data/new_species_2024.txt (6971 chars)
Source used: https://drive.google.com/uc?export=download&id=1A7yJs7OMnEZ3b4His28zZp24PisEYljJ
SOURCE_URL: https://www.nhm.ac.uk/discover/news/2024/december/dicaprios-snake-saurons-piranha-natural-history-museum-describe-190-new-species-2024.html

Over the last 12 months our scientists have been describing a dizzying array of new species. From dinosaurs that may have roamed like cattle along Britain’s ancient waterways to tiny diatoms ceaselessly producing oxygen in the warm waters off Australia’s northern beaches. It has been a year of incredible diversity.

Throughout 2024, our scientis...


## 3) Core Learning Loop: No RAG -> RAG

### RAG Concept In One Minute

RAG has two core steps:
1. Retrieval: find relevant chunks from your documents
2. Generation: answer using those chunks as grounded context

In this section you will see a direct comparison:
- First ask the LLM with no retrieval
- Then ask the same question after indexing the source text

### 2.1 Configure LLM And Embedding Model

We configure two Ollama-backed components:
- an LLM for answer generation
- an embedding model for semantic retrieval

This cell is shared and works in local or Colab once Section 1 is complete.

In [48]:
# Configure LlamaIndex to use local Ollama models
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

LLM_MODEL = "llama3.2:3b"
EMBED_MODEL = "nomic-embed-text"

Settings.llm = Ollama(model=LLM_MODEL, request_timeout=180.0)
Settings.embed_model = OllamaEmbedding(model_name=EMBED_MODEL)
Settings.chunk_size = 512
Settings.chunk_overlap = 50

print(f"LLM configured: {LLM_MODEL}")
print(f"Embedding model configured: {EMBED_MODEL}")

LLM configured: llama3.2:3b
Embedding model configured: nomic-embed-text


### 2.2 The Problem: LLM Knowledge Cutoffs

First, ask the local model a recent 2024 question with no retrieval context.
This establishes the failure mode that RAG solves.

In [27]:
# The same question asked without retrieval context
prompt = "Name me a vegetarian piranha discovered in 2024 and tell me who it was named after."
response = Settings.llm.complete(prompt)

print(f"Query: {prompt}\n")
print("=== Direct LLM response (no RAG) ===")
print(getattr(response, "text", str(response)))

Query: Name me a vegetarian piranha discovered in 2024 and tell me who it was named after.

=== Direct LLM response (no RAG) ===
I don't have any information on a vegetarian piranha discovered in 2024. In fact, piranhas are carnivorous fish that feed on other animals, including fish, crustaceans, and even carrion.

As of my knowledge cutoff in December 2023, there is no scientific evidence to support the existence of a vegetarian piranha. Piranhas have been extensively studied in their natural habitats, and all known species are carnivores, feeding on a variety of prey to survive.

Therefore, I couldn't find any information on a vegetarian piranha discovered in 2024 or any other year, nor could I identify someone it was named after.


### 2.3 How Basic RAG Works

To build basic RAG, we follow four steps:
1. Chunking: split long text into smaller passages.
2. Embedding: convert each chunk into a vector representation of meaning.
3. Retrieval: embed the query and find the most similar chunks.
4. Generation: answer with the query plus retrieved chunks as context.

This is why the same question can perform better after indexing.

In [ ]:
# Shared (Local + Colab): build vector index and rerun the same prompt with retrieval
from pathlib import Path
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

source_file = Path("data/new_species_2024.txt")
if not source_file.exists():
    raise FileNotFoundError(
        "Missing data/new_species_2024.txt. Run Section 0 scraping cell first, "
        "or generate it via scripts/scrape_nhm_new_species.py."
    )

documents = SimpleDirectoryReader(input_files=[str(source_file)]).load_data()
print(f"Loaded {len(documents)} document(s) from {source_file}.")

vector_index = VectorStoreIndex.from_documents(documents)
vector_query_engine = vector_index.as_query_engine(similarity_top_k=3)

# Short aliases used in later sections
index = vector_index
query_engine = vector_query_engine

rag_response = vector_query_engine.query(prompt)
print("=== RAG response ===")
print(rag_response)

Loaded 1 document(s).
=== RAG response ===
The vegetarian piranha discovered in 2024 is Myloplus sauron. It was named after Sauron, the main antagonist from J.R.R. Tolkien's "The Lord of the Rings".


### 2.4 Peek Under The Hood

Inspect the retrieved chunks that grounded the RAG answer.

In [29]:
# Inspect which chunks were retrieved for the RAG answer
for i, node in enumerate(rag_response.source_nodes, 1):
    print(f"--- Retrieved Chunk {i} (Score: {node.score:.4f}) ---")
    print(f"{node.text[:250]}...\n")

--- Retrieved Chunk 1 (Score: 0.7089) ---
“This particular group of pterosaurs is known largely from China,” explains Professor Paul Barrett, one of our dinosaur researchers who helped name the flying reptile. “And so, this is a range extension to another part of the world where we didn’t kn...

--- Retrieved Chunk 2 (Score: 0.6891) ---
SOURCE_URL: https://www.nhm.ac.uk/discover/news/2024/december/dicaprios-snake-saurons-piranha-natural-history-museum-describe-190-new-species-2024.html

Over the last 12 months our scientists have been describing a dizzying array of new species. From...

--- Retrieved Chunk 3 (Score: 0.6716) ---
“We have described this remarkable moth, which we know is from central Guyana, but the only specimens and other materials known in the world are from Port Talbot in South Wales.” It’s joined on our new species list by 11 more moths, including two fro...



### 2.5 Run Diverse RAG Questions

These example queries cover multiple 2024 species discoveries so you can better demonstrate retrieval breadth, not just one fact.

In [22]:
# Intro RAG: run a diverse new-species question set
sample_questions = [
    "Name me a vegetarian piranha",
    "Which species in the NHM 2024 roundup were named after famous figures, and where were they found?",
    "What made the Carmenta brachyclados discovery in South Wales so unusual?",
    "Summarize the range of organism groups in the 2024 roundup (for example moths, beetles, copepods, amphipods) and why this diversity matters.",
    "How does the article connect new-species discovery to conservation and infrastructure decisions?",
]

for i, q in enumerate(sample_questions, 1):
    print(f"\n=== Question {i} ===")
    print(q)
    response = query_engine.query(q)
    print(response)


=== Question 1 ===
Name me a vegetarian piranha
Myloplus sauron.

=== Question 2 ===
Which species in the NHM 2024 roundup were named after famous figures, and where were they found?
There are a few species that were named after famous figures. They are:

* Alococopros milnei (fossil poop) - named after author AA Milne
* Anguiculus dicaprioi (DiCaprio's Himalayan snake) - named after actor and environmentalist Leonard DiCaprio

=== Question 3 ===
What made the Carmenta brachyclados discovery in South Wales so unusual?
The discovery was unusual because it started with a moth found fluttering around a living room in south Wales, but further investigation revealed that it is actually native to the rainforests of Guyana, South America, approximately 7,000 kilometers away.

=== Question 4 ===
Summarize the range of organism groups in the 2024 roundup (for example moths, beetles, copepods, amphipods) and why this diversity matters.
This year's roundup has featured an impressive array of div

## 3) Advanced Retrieval

### Why Basic RAG Is Not Always Enough

Vector retrieval can return chunks that share words but are only loosely relevant.
If too much weak context is passed to the LLM, answer quality can drop.

Reranking helps by scoring the retrieved chunks for relevance and keeping only the strongest evidence before synthesis.

### 3.1 Baseline Retrieval

Start with standard vector retrieval so you have a clear baseline before adding reranking.

In [54]:
# Shared (Local + Colab): baseline retrieval (no reranker)
query = "Which species in the NHM 2024 roundup were named after famous figures, and where were they found?"

baseline_engine = index.as_query_engine(similarity_top_k=8)
baseline_response = baseline_engine.query(query)

print("=== Baseline response ===")
print(baseline_response)

=== Baseline response ===
The Carmenta brachyclados moth was named after a person. It is from South America. The Alococopros milnei fossil was also named after a famous character in literature, Winnie the Pooh, created by author AA Milne.


### 3.2 Add LLM Reranking

Reranking scores initially retrieved chunks and keeps only the strongest evidence before synthesis.

For teaching clarity, this demo uses `top_n=1` so the filtering effect is obvious.
In Section 5, you can increase `top_n` and compare behavior.

In [45]:
# Advanced retrieval: LLM-based reranking
from llama_index.core.postprocessor import LLMRerank

RERANK_TOP_N = 1
reranker = LLMRerank(choice_batch_size=4, top_n=RERANK_TOP_N)
rerank_engine = index.as_query_engine(
    similarity_top_k=12,
    node_postprocessors=[reranker],
)
rerank_response = rerank_engine.query(query)

print("=== Reranked response ===")
print(rerank_response)
print(f"Reranked nodes returned: {len(rerank_response.source_nodes)} (top_n={RERANK_TOP_N})")

=== Reranked response ===
The discovery of the moth Carmenta brachyclados in a living room in south Wales was unusual because it was likely carried there accidentally. The caterpillars of the clearwing moth got stuck to a fragment of seed pod and were transported on the boot of a professional photographer, who then took an aeroplane to the UK.

This species is actually native to the rainforests of Guyana, South America, approximately 7,000 kilometres away from where it was found in Wales.
Reranked nodes returned: 1 (top_n=1)


In [46]:
# Compare retrieved source chunks
print("--- Baseline source nodes ---")
for i, node in enumerate(baseline_response.source_nodes, 1):
    print(f"[{i}] score={node.score:.4f} | {node.text[:120]}...")

print()
print("--- Reranked source nodes ---")
for i, node in enumerate(rerank_response.source_nodes, 1):
    print(f"[{i}] score={node.score:.4f} | {node.text[:120]}...")

--- Baseline source nodes ---
[1] score=0.6963 | SOURCE_URL: https://www.nhm.ac.uk/discover/news/2024/december/dicaprios-snake-saurons-piranha-natural-history-museum-des...
[2] score=0.6748 | “We have described this remarkable moth, which we know is from central Guyana, but the only specimens and other material...
[3] score=0.6525 | “This particular group of pterosaurs is known largely from China,” explains Professor Paul Barrett, one of our dinosaur ...
[4] score=0.6286 | From the islands of Indonesia, our scientists helped to describe four new species of rats, and a new bat has been descri...

--- Reranked source nodes ---
[1] score=10.0000 | SOURCE_URL: https://www.nhm.ac.uk/discover/news/2024/december/dicaprios-snake-saurons-piranha-natural-history-museum-des...


### Interpreting The Source Nodes

When comparing baseline vs reranked source nodes, look for:
- Higher relevance to the query intent
- Less off-topic context
- Better support for key claims in the final answer

In this section, one reranked node is expected because `top_n=1` is intentional for teaching clarity.

## 4) Intent Routing: Your First AI Agent

Fact questions and summary questions often need different retrieval strategies.
Routing lets the LLM choose which tool to use based on the query intent.

This is a foundational agent pattern: tool selection by an LLM policy.
Here we use `RouterQueryEngine` with two tools:
- Vector query engine for specific facts
- Summary query engine for broad overviews

In [32]:
# Native LlamaIndex routing with tool selection
from llama_index.core import SummaryIndex
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool

summary_index = SummaryIndex.from_documents(documents)
summary_query_engine = summary_index.as_query_engine()

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description="Useful for retrieving specific facts, names, or details about individual species discovered in 2024.",
)

summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description="Useful for summarizing the full source, identifying high-level themes, and broad overviews.",
)

router_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[summary_tool, vector_tool],
)

print("Router engine ready.")

Router engine ready.


### 4.1 Test The Router

Run both a summary intent and a specific fact intent, then inspect which tool the router selected.

In [40]:
# Test routing on two different intents
router_queries = [
    "Summarize the main themes of the 2024 new-species discoveries.",
    "What is Myloplus sauron named after?",
]

for q in router_queries:
    print(f"\n--- Query: {q} ---")
    response = router_engine.query(q)
    print("Router selection:", response.metadata.get("selector_result"))
    print("Final answer:")
    print(response)


--- Query: Summarize the main themes of the 2024 new-species discoveries. ---
Router selection: selections=[SingleSelection(index=0, reason='Useful for summarizing the full source, identifying high-level themes, and broad overviews.')]
Final answer:
The discoveries highlight the incredible diversity of life on our planet, from tiny organisms to ancient fossils. The findings emphasize the importance of documentation and naming species to better understand biodiversity, conservation, and the natural world.

These discoveries also underscore the need to protect unique and endemic species, as seen in regions with high species diversity such as the Xingu River in Brazil. Additionally, the examples of species that have been transported across distances by human activity, like the moth from Guyana to Wales, highlight the interconnectedness of ecosystems.

Furthermore, the range extension of known fossil groups, like pterosaurs, and the discovery of new dinosaurs provide insight into the Eart

### 4.2 Try Your Own Routed Prompt

Ask anything about the 2024 discoveries and inspect how the router decides.

In [36]:
# Interactive router demo
custom_query = input("Ask a species question or ask for a summary: ").strip()
if custom_query:
    response = router_engine.query(custom_query)
    print("Router selection:", response.metadata.get("selector_result"))
    print("Final answer:")
    print(response)
else:
    print("No query entered.")

Router selection: selections=[SingleSelection(index=1, reason='Not enough information about specific details of human behavior affecting animals')]
Final answer:
The author suggests that human behavior can have a significant impact on animal species. For instance, the building of dams across the Xingu River in Brazil has put the future of potentially new endemic species at risk due to the gross underestimation of their diversity. This implies that when humans underestimate the number of unique species found in a particular region or area, they may be less likely to take conservation efforts into consideration, which can lead to further threats to these species.

Furthermore, it is mentioned that some species of candiru catfish have an unfortunate habit of swimming up the genitals of unsuspecting swimmers. This suggests that human behavior can not only harm animals but also have serious consequences for human health and well-being.

However, the author emphasizes the importance of docum

### Environment Quick Reference

This notebook is intentionally dual-use: local VS Code/Jupyter or Colab.

Run labels used in code cells:
- `COLAB ONLY`: run only when `IN_COLAB` is true
- `Shared (Local + Colab)`: run in both environments

Practical tips:
- Local is best for repeat practice because models persist
- Colab is best for quick classroom starts but runtime resets are common
- After a Colab reset, rerun Section 1 to reinstall/start Ollama and pull models

## 5) The Tuning Sandbox

RAG is not plug-and-play; it needs tuning.
Use the next cell to change parameters and immediately see how retrieval and answers change.

Things to try:
- Set `CUSTOM_CHUNK_SIZE` to `128` or `1024`
- Set `CUSTOM_TOP_K` to `2`
- Change `TEST_QUERY` to another species question

### Why This Cell Is Useful

The parameters are exposed at the top so you can see exactly where each one affects the LlamaIndex pipeline.
Change values, rerun, and compare the final answer and retrieved nodes.

In [50]:
from llama_index.core.postprocessor import LLMRerank

# ==========================================
# 1. TINKER WITH THESE VARIABLES
# ==========================================
CUSTOM_CHUNK_SIZE = 512
CUSTOM_CHUNK_OVERLAP = 50
CUSTOM_TOP_K = 10         # Initial vector retrieval breadth
CUSTOM_TOP_N_RERANK = 2   # Chunks kept after reranking

TEST_QUERY = "Why was the Carmenta brachyclados discovery in south Wales unusual?"

# ==========================================
# 2. THE RAG PIPELINE (direct LlamaIndex API)
# ==========================================
Settings.chunk_size = CUSTOM_CHUNK_SIZE
Settings.chunk_overlap = CUSTOM_CHUNK_OVERLAP

print("Re-building index with your custom chunk settings...")
sandbox_index = VectorStoreIndex.from_documents(documents)

sandbox_reranker = LLMRerank(choice_batch_size=4, top_n=CUSTOM_TOP_N_RERANK)
sandbox_engine = sandbox_index.as_query_engine(
    similarity_top_k=CUSTOM_TOP_K,
    node_postprocessors=[sandbox_reranker],
)

# ==========================================
# 3. RESULTS
# ==========================================
print(f"\n--- Query: {TEST_QUERY} ---")
sandbox_response = sandbox_engine.query(TEST_QUERY)

print("\nFinal Answer:")
print(sandbox_response)

print(f"\n--- Nodes Used (up to {CUSTOM_TOP_N_RERANK}) ---")
for i, node in enumerate(sandbox_response.source_nodes, 1):
    print(f"\n[Node {i} | Score: {node.score:.4f}]")
    print(f"{node.text[:220]}...")

Re-building index with your custom chunk settings...

--- Query: Why was the Carmenta brachyclados discovery in south Wales unusual? ---

Final Answer:
The discovery of the Carmenta brachyclados moth in south Wales was unusual because it was found fluttering around a living room, which is an unexpected place for such a species.

--- Nodes Used (up to 2) ---

[Node 1 | Score: 9.0000]
SOURCE_URL: https://www.nhm.ac.uk/discover/news/2024/december/dicaprios-snake-saurons-piranha-natural-history-museum-describe-190-new-species-2024.html

Over the last 12 months our scientists have been describing a dizzy...

[Node 2 | Score: 8.0000]
“This particular group of pterosaurs is known largely from China,” explains Professor Paul Barrett, one of our dinosaur researchers who helped name the flying reptile. “And so, this is a range extension to another part o...
